# Part 1: Hadoop & MapReduce
### Banking Dataset Analysis
> Run this notebook in Google Colab. All code uses pure Python MapReduce simulation (no Hadoop installation needed in Colab).

In [ ]:
# ── Install dependencies
!pip install pandas matplotlib seaborn -q
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from collections import defaultdict
import csv, io
print('Libraries loaded ✓')

In [ ]:
# ── Upload bank.csv  (run this cell, then click 'Choose Files')
from google.colab import files
uploaded = files.upload()  # upload bank.csv
df = pd.read_csv('bank.csv', sep=',')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## Q1 – Data Ingestion
Simulating HDFS directory creation and data transfer.

In [ ]:
# Simulate HDFS ingestion (local)
import os
os.makedirs('hdfs_sim/banking', exist_ok=True)
df.to_csv('hdfs_sim/banking/bank.csv', index=False)
files_in_hdfs = os.listdir('hdfs_sim/banking')
print('HDFS directory: hdfs_sim/banking/')
print('Files:', files_in_hdfs)
print(f'File size: {os.path.getsize("hdfs_sim/banking/bank.csv")/1024:.1f} KB')

## Q2 – MapReduce: Average Balance per Job Type

In [ ]:
# MapReduce Job 1: Average account balance per job type
# --- MAP phase ---
mapped = [(row['job'], row['balance']) for _, row in df.iterrows()]

# --- SHUFFLE & SORT phase ---
from collections import defaultdict
shuffled = defaultdict(list)
for job, bal in mapped:
    shuffled[job].append(bal)

# --- REDUCE phase ---
result_mr1 = {job: round(sum(vals)/len(vals), 2) for job, vals in shuffled.items()}

# Display results
mr1_df = pd.DataFrame(list(result_mr1.items()), columns=['Job', 'Avg_Balance'])
mr1_df = mr1_df.sort_values('Avg_Balance', ascending=False).reset_index(drop=True)
print('MapReduce Result: Average Balance per Job Type')
print(mr1_df.to_string())

In [ ]:
# Visualize MR1 result
plt.figure(figsize=(12,5))
plt.bar(mr1_df['Job'], mr1_df['Avg_Balance'], color='steelblue', edgecolor='black')
plt.xticks(rotation=45, ha='right')
plt.title('Average Account Balance by Job Type (MapReduce)', fontsize=14)
plt.ylabel('Average Balance')
plt.tight_layout()
plt.savefig('mr1_avg_balance_by_job.png', dpi=150)
plt.show()

## Q2b – MapReduce: Housing Loan Count per Education Category

In [ ]:
# MapReduce Job 2: Housing loan count per education
mapped2 = [((row['education'], row['housing']), 1) for _, row in df.iterrows()]
shuffled2 = defaultdict(int)
for key, val in mapped2:
    shuffled2[key] += val

result_mr2 = pd.DataFrame(
    [(edu, housing, count) for (edu, housing), count in shuffled2.items()],
    columns=['Education', 'Housing_Loan', 'Count']
).sort_values(['Education', 'Housing_Loan'])
print('MapReduce Result: Housing Loan Count per Education Category')
print(result_mr2.to_string(index=False))

In [ ]:
# Pivot and visualize
pivot = result_mr2.pivot(index='Education', columns='Housing_Loan', values='Count').fillna(0)
pivot.plot(kind='bar', figsize=(10,5), edgecolor='black')
plt.title('Housing Loan Distribution by Education Level')
plt.xlabel('Education')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.legend(title='Housing Loan')
plt.tight_layout()
plt.savefig('mr2_housing_by_education.png', dpi=150)
plt.show()

## Q2c – MapReduce: Contacts per Month by Subscription Status

In [ ]:
# MapReduce Job 3
mapped3 = [((row['month'], row['y']), 1) for _, row in df.iterrows()]
shuffled3 = defaultdict(int)
for key, val in mapped3:
    shuffled3[key] += val

month_order = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
result_mr3 = pd.DataFrame(
    [(m, y, c) for (m,y), c in shuffled3.items()],
    columns=['Month','Subscribed','Count']
)
result_mr3['month_idx'] = result_mr3['Month'].map({m:i for i,m in enumerate(month_order)})
result_mr3 = result_mr3.sort_values('month_idx')
print(result_mr3[['Month','Subscribed','Count']].to_string(index=False))

## Q3a – MapReduce: Avg Contact Duration per Campaign Outcome

In [ ]:
# MapReduce Job 4: Avg duration per poutcome
mapped4 = [(row['poutcome'], row['duration']) for _, row in df.iterrows()]
s4 = defaultdict(lambda: [0,0])
for pout, dur in mapped4:
    s4[pout][0] += dur; s4[pout][1] += 1
result_mr4 = pd.DataFrame(
    [(k, round(v[0]/v[1],2), v[1]) for k,v in s4.items()],
    columns=['poutcome','avg_duration_sec','count']
).sort_values('avg_duration_sec', ascending=False)
print('Average Contact Duration per Campaign Outcome:')
print(result_mr4.to_string(index=False))

## Q3b – MapReduce: Age vs Balance Relationship

In [ ]:
# MapReduce Job 5: Age vs balance by decade
mapped5 = [(f"{(row['age']//10)*10}s", row['balance']) for _, row in df.iterrows()]
s5 = defaultdict(lambda: [0,0,float('inf'),float('-inf')])
for band, bal in mapped5:
    s5[band][0]+=bal; s5[band][1]+=1
    s5[band][2]=min(s5[band][2],bal); s5[band][3]=max(s5[band][3],bal)
result_mr5 = pd.DataFrame(
    [(b, round(v[0]/v[1],2), v[1], v[2], v[3]) for b,v in sorted(s5.items())],
    columns=['Age_Band','Avg_Balance','Count','Min_Balance','Max_Balance']
)
print('Age vs Balance Relationship:')
print(result_mr5.to_string(index=False))

plt.figure(figsize=(9,4))
plt.plot(result_mr5['Age_Band'], result_mr5['Avg_Balance'], marker='o', color='darkblue')
plt.title('Average Balance by Age Decade')
plt.xlabel('Age Band'); plt.ylabel('Avg Balance')
plt.grid(True); plt.tight_layout()
plt.savefig('mr5_age_vs_balance.png', dpi=150)
plt.show()